# Sprint 3 — Feature Engineering y Pipeline (v2)
## Proyecto: Productividad Asesores de Negocios

---

### Cambios respecto a v1

- 29 features en lugar de 22
- 4 variables de evento nuevas incluidas como numéricas:
  `TASA_DESEMBOLSO`, `CLIENTES_PRESTAMO_EVENTO`, `MONTO_DESEMBOLSO_EVENTO`, `CLIENTES_NUEVOS_EVENTO`
- Variables originales de evento excluidas: `CLIENTES_PRESTAMO`, `PRESTAMO`, `CLIENTES_NUEVOS`, `DESEMBOLSO_CLIENTES_NUEVOS`
- Target con 3 clases: BAJO, MEDIO, ALTO


## 1. Configuración y carga de datos

In [1]:
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import joblib

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, LabelEncoder
from sklearn.impute import SimpleImputer

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

INTERIM   = Path('../data/interim')
PROCESSED = Path('../data/processed')
PROCESSED.mkdir(parents=True, exist_ok=True)

df_train = pd.read_parquet(INTERIM / 'target_b_train.parquet')
df_test  = pd.read_parquet(INTERIM / 'target_b_test.parquet')

print(f'Train: {df_train.shape}')
print(f'Test : {df_test.shape}')
print(f'Columnas disponibles:')
print(sorted(df_train.columns.tolist()))


Train: (1603, 31)
Test : (401, 31)
Columnas disponibles:
['AREA', 'CLIENTES', 'CLIENTES_NUEVOS_EVENTO', 'CLIENTES_PRESTAMO_EVENTO', 'CODIGO_ASESOR', 'CODIGO_SUCURSAL', 'COORDINADOR', 'DIA_PAGO', 'EDAD', 'ESTADO_CIVIL', 'EXP_MICROFINANZAS', 'FECHA_ALTA', 'FECHA_BAJA', 'FECHA_NACIMIENTO', 'GRUPOS', 'INCREMENTO_CARTERA', 'MONTO_DESEMBOLSO_EVENTO', 'MOTIVO_BAJA', 'NIVEL_ESTUDIOS', 'N_SEMANAS_OBS', 'PRODUCTO', 'PUESTO', 'RANGO_CICLO', 'REGION', 'SEXO', 'SUCURSAL', 'TARGET_B', 'TASA_DESEMBOLSO', 'TASA_PROM', 'TDNC', 'TIPO_BAJA']


## 2. Limpieza previa al pipeline

In [2]:
def limpiar_previo(df):
    '''
    Limpieza previa al pipeline de sklearn.
    Corrige problemas de calidad detectados en el EDA.
    '''
    df = df.copy()
    df['TIPO_BAJA']         = df['TIPO_BAJA'].replace('', 'ACTIVO').fillna('ACTIVO')
    df['EXP_MICROFINANZAS'] = df['EXP_MICROFINANZAS'].replace('', 'SD').fillna('SD')
    df['MOTIVO_BAJA']       = df['MOTIVO_BAJA'].replace('', 'ACTIVO').fillna('ACTIVO')
    if 'NIVEL_ESTUDIOS' in df.columns:
        df['NIVEL_ESTUDIOS'] = df['NIVEL_ESTUDIOS'].fillna('SD').str.upper().str.strip()
    if 'ESTADO_CIVIL' in df.columns:
        df['ESTADO_CIVIL'] = df['ESTADO_CIVIL'].fillna('SD').str.upper().str.strip()
    if 'SEXO' in df.columns:
        df['SEXO'] = df['SEXO'].fillna('SD').str.upper().str.strip()
    return df


df_train = limpiar_previo(df_train)
df_test  = limpiar_previo(df_test)

print('Verificacion post-limpieza:')
for col in ['TIPO_BAJA', 'EXP_MICROFINANZAS', 'NIVEL_ESTUDIOS', 'ESTADO_CIVIL', 'SEXO']:
    if col in df_train.columns:
        vacios = (df_train[col] == '').sum()
        nulos  = df_train[col].isna().sum()
        print(f'  {col:<25} vacios={vacios} | nulos={nulos}')


Verificacion post-limpieza:
  TIPO_BAJA                 vacios=0 | nulos=0
  EXP_MICROFINANZAS         vacios=0 | nulos=0
  NIVEL_ESTUDIOS            vacios=0 | nulos=0
  ESTADO_CIVIL              vacios=0 | nulos=0
  SEXO                      vacios=0 | nulos=0


## 3. Definición estática de columnas

Las 4 variables de evento nuevas son numéricas continuas y se procesan igual que el resto de numéricas.


In [3]:
# ── Variables numéricas ───────────────────────────────────────────────────────
NUM_COLS = [
    # Variables operacionales de estado continuo
    'GRUPOS',
    'CLIENTES',
    'TASA_PROM',
    'INCREMENTO_CARTERA',
    'N_SEMANAS_OBS',
    'EDAD',                          # FAIRNESS
    # Variables de evento correctamente calculadas (v3)
    'TASA_DESEMBOLSO',               # Actividad de renovacion de cartera
    'CLIENTES_PRESTAMO_EVENTO',      # Tamano promedio del grupo al desembolso
    'MONTO_DESEMBOLSO_EVENTO',       # Monto promedio desembolsado por grupo
    'CLIENTES_NUEVOS_EVENTO',        # Clientes nuevos promedio en desembolsos
]

# ── Variables categóricas ─────────────────────────────────────────────────────
CAT_COLS = [
    'REGION',
    'RANGO_CICLO',
    'DIA_PAGO',
    'PUESTO',
    'PRODUCTO',
    'AREA',
    'TIPO_BAJA',
    'MOTIVO_BAJA',
    'EXP_MICROFINANZAS',
    'NIVEL_ESTUDIOS',
    'SEXO',                          # FAIRNESS
    'ESTADO_CIVIL',                  # FAIRNESS
]

# Verificar que todas las columnas existen
todas = NUM_COLS + CAT_COLS
faltantes = [c for c in todas if c not in df_train.columns]
if faltantes:
    print(f'ADVERTENCIA columnas faltantes: {faltantes}')
else:
    print(f'Columnas numericas  : {len(NUM_COLS)}')
    print(f'Columnas categoricas: {len(CAT_COLS)}')
    print(f'Total features      : {len(todas)}')
    print(f'Variables de evento : TASA_DESEMBOLSO, CLIENTES_PRESTAMO_EVENTO,')
    print(f'                      MONTO_DESEMBOLSO_EVENTO, CLIENTES_NUEVOS_EVENTO')


Columnas numericas  : 10
Columnas categoricas: 12
Total features      : 22
Variables de evento : TASA_DESEMBOLSO, CLIENTES_PRESTAMO_EVENTO,
                      MONTO_DESEMBOLSO_EVENTO, CLIENTES_NUEVOS_EVENTO


## 4. Construcción del pipeline

In [4]:
def build_pipeline():
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
    ])
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='SD')),
        ('encoder', OrdinalEncoder(
            handle_unknown='use_encoded_value',
            unknown_value=-1,
        )),
    ])
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer,    NUM_COLS),
            ('cat', categorical_transformer, CAT_COLS),
        ],
        remainder='drop',
        verbose_feature_names_out=True,
    )
    return preprocessor


preprocessor = build_pipeline()
print('Pipeline construido correctamente.')
print(f'Transformers: {[t[0] for t in preprocessor.transformers]}')


Pipeline construido correctamente.
Transformers: ['num', 'cat']


## 5. Separar X e y — fitear en train

In [5]:
TARGET_COL   = 'TARGET_B'
feature_cols = [c for c in NUM_COLS + CAT_COLS if c in df_train.columns]

X_train = df_train[feature_cols].copy()
y_train = df_train[TARGET_COL].copy()
X_test  = df_test[feature_cols].copy()
y_test  = df_test[TARGET_COL].copy()

# Codificar target: ALTO=0, BAJO=1, MEDIO=2
label_encoder = LabelEncoder()
y_train_enc   = label_encoder.fit_transform(y_train)
y_test_enc    = label_encoder.transform(y_test)

print(f'X_train: {X_train.shape}')
print(f'X_test : {X_test.shape}')
print(f'Clases : {list(label_encoder.classes_)}')
print(f'Encoding: {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}')

# PASO 1: fitear SOLO en train
X_train_processed = preprocessor.fit_transform(X_train)
# PASO 2: aplicar pipeline ya fiteado a test (sin recalcular)
X_test_processed  = preprocessor.transform(X_test)

feature_names_out = preprocessor.get_feature_names_out()

print(f'\nX_train procesado: {X_train_processed.shape}')
print(f'X_test  procesado: {X_test_processed.shape}')
print(f'Features de salida: {len(feature_names_out)}')

nan_train = np.isnan(X_train_processed).sum()
nan_test  = np.isnan(X_test_processed).sum()
print(f'NaN en train procesado: {nan_train}')
print(f'NaN en test  procesado: {nan_test}')


X_train: (1603, 22)
X_test : (401, 22)
Clases : ['ALTO', 'BAJO', 'MEDIO']
Encoding: {'ALTO': np.int64(0), 'BAJO': np.int64(1), 'MEDIO': np.int64(2)}

X_train procesado: (1603, 22)
X_test  procesado: (401, 22)
Features de salida: 22
NaN en train procesado: 0
NaN en test  procesado: 0


## 6. Tests de integridad del pipeline

In [6]:
print('=== VERIFICACIONES DE INTEGRIDAD ===')

assert X_train_processed.shape[1] == X_test_processed.shape[1]
print(f'[OK] Test 1: mismas features train y test ({X_train_processed.shape[1]})')

assert np.isnan(X_train_processed).sum() == 0
assert np.isnan(X_test_processed).sum()  == 0
print('[OK] Test 2: sin NaN en train ni test procesados')

mediana_train = preprocessor.named_transformers_['num']['imputer'].statistics_[
    NUM_COLS.index('CLIENTES')
]
mediana_real = np.nanmedian(X_train['CLIENTES'])
assert abs(mediana_train - mediana_real) < 1.0
print(f'[OK] Test 3: mediana de CLIENTES en pipeline = {mediana_train:.2f} (calculada en train)')

clases_train = set(y_train_enc)
clases_test  = set(y_test_enc)
assert clases_train == clases_test
print(f'[OK] Test 4: mismas clases en train y test: {sorted(clases_train)}')

cat_start = len(NUM_COLS)
n_unknowns = (X_train_processed[:, cat_start:] == -1).sum()
if n_unknowns > 0:
    print(f'[WARN] Test 5: {n_unknowns} valores unknown en categoricas de train')
else:
    print('[OK] Test 5: sin valores unknown en categoricas de train')

print('\nTodos los tests pasaron correctamente.')


=== VERIFICACIONES DE INTEGRIDAD ===
[OK] Test 1: mismas features train y test (22)
[OK] Test 2: sin NaN en train ni test procesados
[OK] Test 3: mediana de CLIENTES en pipeline = 9.68 (calculada en train)
[OK] Test 4: mismas clases en train y test: [np.int64(0), np.int64(1), np.int64(2)]
[OK] Test 5: sin valores unknown en categoricas de train

Todos los tests pasaron correctamente.


## 7. Guardar pipeline y datos procesados

In [7]:
joblib.dump(preprocessor,  PROCESSED / 'preprocessor.joblib')
joblib.dump(label_encoder, PROCESSED / 'label_encoder.joblib')

np.savez(
    PROCESSED / 'features_processed.npz',
    X_train=X_train_processed,
    X_test=X_test_processed,
    y_train=y_train_enc,
    y_test=y_test_enc,
    feature_names=feature_names_out,
)

pd.Series(feature_names_out).to_csv(PROCESSED / 'feature_names.csv', index=False)

print('Archivos guardados en data/processed/:')
for f in sorted(PROCESSED.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<40} {size_kb:>8.1f} KB')

print(f'\nResumen final:')
print(f'  Features entrada : {len(feature_cols)}')
print(f'  Features salida  : {X_train_processed.shape[1]}')
print(f'  Train            : {X_train_processed.shape[0]:,} asesores')
print(f'  Test             : {X_test_processed.shape[0]:,} asesores')
print(f'  Clases del target: {list(label_encoder.classes_)}')


Archivos guardados en data/processed/:
  .gitkeep                                      0.0 KB
  feature_names.csv                             0.4 KB
  features_processed.npz                      361.9 KB
  label_encoder.joblib                          0.5 KB
  modelo_final.joblib                         599.7 KB
  preprocessor.joblib                           7.5 KB

Resumen final:
  Features entrada : 22
  Features salida  : 22
  Train            : 1,603 asesores
  Test             : 401 asesores
  Clases del target: ['ALTO', 'BAJO', 'MEDIO']
